In [ ]:
# 1/4 - clone main + prototype + Inference, install .NET (fault-tolerant)
import subprocess, os, shutil
print('===== INTEGRATE 14 PURE TKV KERNELS INTO NATIVE BUILD + LOSS CHECK =====')
for d,br in [('/content/TVmain','main'),('/content/TVcpu','prototype/cuda-backend')]:
    subprocess.run('rm -rf '+d, shell=True)
    subprocess.run('git clone --depth 1 --branch '+br+' https://github.com/nguyenhungtran18/TokenVector.git '+d, shell=True, capture_output=True, text=True, timeout=600)
    print(br, subprocess.check_output('git -C '+d+' rev-parse --short HEAD', shell=True).decode().strip())
subprocess.run('rm -rf /content/TVinf', shell=True)
subprocess.run('git clone --depth 1 https://github.com/nguyenhungtran18/TokenVector.Inference.git /content/TVinf', shell=True, capture_output=True, text=True, timeout=600)
dotnet=shutil.which('dotnet')
if not dotnet:
    print('dotnet not found -> installing .NET 10 SDK...')
    p=subprocess.run('wget -q https://dot.net/v1/dotnet-install.sh -O /tmp/dotnet-install.sh && chmod +x /tmp/dotnet-install.sh && /tmp/dotnet-install.sh --channel 10.0 --install-dir /usr/share/dotnet', shell=True, capture_output=True, text=True, timeout=900)
    print('install rc=',p.returncode)
    dotnet='/usr/share/dotnet/dotnet'
print('dotnet =',dotnet,'exists=',os.path.exists(dotnet))

In [ ]:
# 2/4 - thay 14 native kernel C bang .cpu.c sinh tu .tkv THUAN (KHONG chen C)
import subprocess, shutil, os, re
mainroot='/content/TVmain/Release/tools/simd_native'
ex='/content/TVcpu/Release/cuda_backend/examples'
# 10 kernel trong tkv_bwd_ops.c -> gom thanh 1 file thay the
bwd=['bias_add_rows_cpu.cpu.c','softmax_causal_fwd_pure.cpu.c','merge_hb_head_cpu.cpu.c',
     'layernorm_bwd_cpu.cpu.c','gelu_erf_bwd_pure.cpu.c','gelu_erf_fwd_pure.cpu.c',
     'bias_grad_cpu.cpu.c','softmax_causal_bwd_pure.cpu.c','embed_bwd_pure.cpu.c',
     'layernorm_fwd_cpu.cpu.c']
parts=[]
for f in bwd:
    s=open(ex+'/'+f).read()
    ls=[l for l in s.split('\n') if not l.startswith('#include') and not l.startswith('#define') and l.strip()!='']
    parts.append('\n'.join(ls))
hdr='#include <immintrin.h>\n#include <math.h>\n#include <stdlib.h>\n#define EXPORT __attribute__((visibility("default")))\n'
open(mainroot+'/tkv_bwd_ops.c','w').write(hdr+'\n\n'.join(parts)+'\n')
print('bwd_ops replaced, bytes=',os.path.getsize(mainroot+'/tkv_bwd_ops.c'),'kernels=',len(bwd))
# 2 kernel file rieng
shutil.copyfile(ex+'/linear_dw_cpu.cpu.c', mainroot+'/tkv_linear_dw.c')
shutil.copyfile(ex+'/linear_dx_cpu.cpu.c', mainroot+'/tkv_linear_dx.c')
print('linear_dw/dx replaced')
# 2 list-op nam trong tkv_simd_native.c -> thay body bang regex
p=mainroot+'/tkv_simd_native.c'; s=open(p).read()
for name,gf in [('F32ListAdd','f32_list_add_cpu.cpu.c'),('F32ListCopy','f32_list_copy_cpu.cpu.c')]:
    pat=re.compile(r'EXPORT int '+name+r'\(.*?\n\}\n', re.S)
    rep=open(ex+'/'+gf).read().strip()+'\n'
    s,n=pat.subn(rep,s,count=1)
    print('patched',name,'count=',n)
open(p,'w').write(s)
# build native .so
srcs='tkv_simd_native.c tkv_cuda_native.c tkv_linear_dw.c tkv_bwd_ops.c tkv_flash_attn.c tkv_step124.c tkv_prof.c tkv_attn_cuda.c'
r=subprocess.run('cd '+mainroot+' && gcc -O2 -mavx2 -mfma -shared -fPIC '+srcs+' -lm -lpthread -ldl -o tkv_simd_native.so', shell=True, capture_output=True, text=True, timeout=600)
print('native build rc=',r.returncode)
if r.returncode!=0: print(r.stderr[:3000])

In [ ]:
# 3/4 - kiem tra 14 symbols + chay native fullstep MODE 2 (mode 1 bi MissingMethodException)
import ctypes, subprocess, os, shutil
so='/content/TVmain/Release/tools/simd_native/tkv_simd_native.so'
lib=ctypes.CDLL(so)
names=['LinearDwFused','BiasAddRows','BiasGrad','MergeHbHead','LayernormFwd',
       'LayernormBwdFused','F32ListAdd','F32ListCopy','LinearDxFused',
       'SoftmaxCausalBwd','SoftmaxCausalFwd','GeluErfFwd','GeluErfBwd','EmbedBwdFused']
missing=[n for n in names if not hasattr(lib,n)]
print('14 symbols missing =',missing)
assert not missing, 'build that bai - thieu symbol'
shutil.copyfile(so,'/content/TVinf/demo/tkv_simd_native.so')
cfg='/content/TVinf/tv/tests/_seq_cfg.txt'
open(cfg,'w').write('1024\n2\n1\n')
dotnet=shutil.which('dotnet') or '/usr/share/dotnet/dotnet'
env=dict(os.environ); env.pop('TKV_TORCH_GPU',None)
cmd='cd /content/TVinf && LD_LIBRARY_PATH=/content/TVinf/demo:/usr/local/lib/python3.13/dist-packages/torch/lib '+dotnet+' demo/bseq.dll'
p=subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=1800, env=env)
print('bseq rc=',p.returncode)
print('STDOUT tail:',(p.stdout or '')[-2200:])
print('STDERR head:',(p.stderr or '')[:600])

In [ ]:
# 4/4 - summary
print('EXPECTED native loss (mode 2) = 25.45386. If differs -> integrated .tkv kernels changed native path.')